# Features and similarity search

Every model computes a representation before it computes an answer. Drop the head, read that tensor instead of the
logits, and a pretrained checkpoint becomes a frozen **encoder**: no labels, no training, and everything downstream is
a dot product.

This notebook builds two searches on top of that idea:

- **inside one scene**: a feature per point $(N, C)$, and the points whose feature is closest to one query point,
- **across a dataset**: one global descriptor per object $(B, C)$, an index over a whole test split, and
  nearest-neighbor retrieval.

The first half runs from a sample room committed with the docs, so it works on Colab with nothing but a checkpoint
download. The second half needs the ModelNet40 benchmark data and says so where it starts.

New here? [Quickstart](01-quickstart.md) covers `create_model` and the packed-batch format, and
[Segment a scene](02-segmentation-inference.md) covers the transform a checkpoint carries with it. The
[Feature maps](../models/features.md) guide is the reference page behind this tutorial.

In [ ]:
# On Colab: !pip install "torch-pointcloud[pyg-lib]"
import torch
import torch.nn.functional as F

import torch_pointcloud as tp

torch.manual_seed(0)
device = "cuda" if torch.cuda.is_available() else "cpu"
print("torch-pointcloud", tp.__version__, "| device:", device)

## Read the representation, not the logits

`num_classes=0` replaces a model's head with `nn.Identity`, so `forward` returns the tensor the head would have
consumed. It works on every model in the registry, needs no knowledge of the architecture, and with `pretrained=True`
the whole backbone still loads: only the head keys are skipped.

What comes back depends on the task:

| task | returns | one row per |
| --- | --- | --- |
| `segmentation` | $(N, C)$ | point |
| `classification` | $(B, C)$ | cloud |

`return_info=True` also hands back the registry entry, whose `transform` is the exact preprocessing the checkpoint was
trained with. Both models below are built with random weights, so this cell runs anywhere in about a second on CPU.

In [ ]:
encoder = tp.create_model("pointnext-sm", task="segmentation", in_channels=4, num_classes=0).eval()
classifier = tp.create_model("pointnet2-ssg.modelnet40.xu-yan", task="classification", num_classes=0).eval()

pos = torch.rand(8192, 3) * 4.0
x = torch.rand(8192, 4)
batch = torch.arange(2).repeat_interleave(4096)

with torch.no_grad():
    per_point = encoder(x, pos, batch)
    per_cloud = classifier(None, pos, batch)

print("per point:", tuple(per_point.shape), "| head:", type(encoder.head).__name__)
print("per cloud:", tuple(per_cloud.shape), "| head:", type(classifier.head).__name__)

```text
per point: (8192, 64) | head: Identity
per cloud: (2, 1024) | head: Identity
```

Two clouds of 4096 points each: the segmentation backbone answers with one 64-channel feature per point, the
classifier pools every cloud into a single 1024-channel descriptor. The rest of the notebook is those two shapes with
real weights behind them.

## Look at what the encoder learned

The room below is `sample_scene_labeled.ply`, a 2.9 MB ScanNet scene committed with the docs: 127 410 points with
color and NYU40 labels. The encoder is `concerto-large-lp.scannet20.pointcept`, a self-supervised checkpoint that was pretrained
without labels. Its linear probe is never called here, and neither are the room's labels until the next section, where
they are used only to score an answer.

In [ ]:
import urllib.request
from pathlib import Path

import numpy as np
from plyfile import PlyData

import torch_pointcloud.transforms as T

path = Path("../assets/data/sample_scene_labeled.ply")  # in a docs checkout
if not path.exists():
    path = Path("sample_scene_labeled.ply")
    url = "https://github.com/arthurdjn/pytorch-pointcloud/raw/main/docs/assets/data/sample_scene_labeled.ply"
    if not path.exists():
        urllib.request.urlretrieve(url, path)

ply = PlyData.read(path)["vertex"]
room = {
    "pos": torch.from_numpy(np.stack([ply["x"], ply["y"], ply["z"]], 1).astype("float32")),
    "color": torch.from_numpy(np.stack([ply["red"], ply["green"], ply["blue"]], 1).astype("float32")),  # 0-255
    "segment": torch.from_numpy(np.asarray(ply["segment"]).astype("int64")),
}
room = T.EstimateNormals(keys="pos", k=16, orient_to_centroid=True)(room)  # the checkpoint reads color + normals
print({key: tuple(value.shape) for key, value in room.items()})

In [ ]:
from torch_pointcloud.utils.data import collate

model, info = tp.create_model(
    "concerto-large-lp.scannet20.pointcept",
    task="segmentation",
    pretrained=True,
    num_classes=0,
    return_info=True,
)
model = model.eval().to(device)

sample = collate([info["transform"](dict(room))])
with torch.inference_mode():
    feat = model(sample["x"].to(device), sample["pos_grid"].to(device), sample["batch"].to(device)).float()

print("features:", tuple(feat.shape), "| voxels:", len(sample["pos"]), "of", len(room["pos"]), "input points")

```text
features: (114118, 1728) | voxels: 114118 of 127410 input points
```

Two things to read off that line. **Features follow the preprocessing**: this checkpoint voxelizes at 2 cm, so there
is one feature per voxel, not per input point. When you need them back at full resolution, set `dst_inverse_key` on
the `Voxelize` in your own pipeline and gather with the index it writes. And $C = 1728$ because the decoder
concatenates every scale on its way back up, so a single row already carries what the encoder saw at each level.

Now project each of those 1728-dimensional rows onto its top principal components and read three numbers as RGB.
Points with similar features get similar colors, with no labels involved:

In [ ]:
def pca_color(feat: torch.Tensor) -> torch.Tensor:
    """Map a per-point feature (N, C) to RGB in [0, 1] through its top principal components."""
    _, _, components = torch.pca_lowrank(feat, center=True, q=6, niter=5)
    projected = feat @ components
    projected = projected[:, :3] * 0.6 + projected[:, 3:6] * 0.4
    low, high = projected.min(0, keepdim=True).values, projected.max(0, keepdim=True).values
    return ((projected - low) / (high - low).clamp_min(1e-6)).clamp(0, 1)


rgb = pca_color(feat).cpu()  # matplotlib draws from host memory
print("rgb:", tuple(rgb.shape), "| range:", (float(rgb.min()), float(rgb.max())))

In [ ]:
import matplotlib.pyplot as plt

voxel = sample["pos"] * 0.02  # the transform voxelizes at 2 cm: grid indices back to meters


def show_clouds(clouds, titles, point_size=1.0, columns=None, height=4.4):
    """Draw one cloud per panel, colored by its own `color`: RGB rows in [0, 1], or one color for the panel."""
    columns = columns or len(clouds)
    rows = -(-len(clouds) // columns)
    _, axes = plt.subplots(rows, columns, figsize=(4.4 * columns, height * rows), subplot_kw={"projection": "3d"})
    for ax, cloud, title in zip(np.ravel(axes), clouds, titles):
        pos = np.asarray(cloud["pos"])
        ax.scatter(*pos.T, c=cloud["color"], s=point_size, linewidths=0, depthshade=False)
        ax.set_box_aspect(np.ptp(pos, axis=0))
        ax.set_title(title, fontsize=10)
        ax.set_axis_off()
    plt.show()


show_clouds(
    [{"pos": room["pos"], "color": room["color"] / 255}, {"pos": voxel, "color": rgb}],
    ["input: color from the scanner", f"features: {feat.shape[1]} channels per point, PCA to RGB"],
    point_size=0.4,
)

![The sample room in its captured color, beside the same points colored by the top principal components of their features.](../assets/tutorials/pca_room.png)

Look at the right panel with the labels in mind: floor, walls, table and chairs each take their own color, and the chairs all take the *same* color as each other. That grouping is what a similarity search is about to exploit.

The hues themselves mean nothing: principal components are defined up to a rotation, so a rerun can recolor the whole room and still group it the same way.

## Query one point

Normalize the rows to unit length and a dot product is a cosine. Pick one point, dot its feature against every other
point, and read the result as a heat map over the room.

The labels enter only to score what came back. The `segment` used below is the voxelized one that the transform wrote
next to the features, so it lines up row for row with `feat`.

In [ ]:
classes = list(info["weights"]["classes"])
segment = sample["segment"].numpy()  # voxelized labels, used to score the answer and nothing else
normalized = F.normalize(feat, dim=-1)  # unit rows, so a dot product is a cosine

chairs = np.where(segment == classes.index("chair"))[0]
query = int(chairs[len(chairs) // 2])  # a point in the middle of the class, not on its boundary
similarity = (normalized[query] @ normalized.t()).cpu()  # (N,) in [-1, 1]

print(f"query {query}: one of {len(chairs)} chair points")
print(f"similarity: min {float(similarity.min()):.2f}, mean {float(similarity.mean()):.2f}")
for k in (100, 1000, 5000):
    top = similarity.topk(k).indices.numpy()
    share = (segment[top] == classes.index("chair")).mean()
    print(f"top {k:>5}: {share:6.1%} chair, lowest cosine {float(similarity[top].min()):.2f}")

found, counts = np.unique(segment[similarity.topk(5000).indices.numpy()], return_counts=True)
print("top  5000:", {("unlabeled" if label < 0 else classes[label]): int(count) for label, count in zip(found, counts)})

```text
query 72863: one of 15115 chair points
similarity: min 0.07, mean 0.24
top   100: 100.0% chair, lowest cosine 0.87
top  1000: 100.0% chair, lowest cosine 0.81
top  5000:  98.0% chair, lowest cosine 0.66
top  5000: {'unlabeled': 83, 'floor': 16, 'chair': 4901}
```

In [ ]:
top = similarity.topk(5000).indices.numpy()
scaled = ((similarity - similarity.min()) / (1 - similarity.min())).numpy()
found = np.unique(segment[top])
palette = {label: plt.get_cmap("tab10")(rank) for rank, label in enumerate(found)}

_, (heat, neighbors) = plt.subplots(1, 2, figsize=(11.0, 4.6), subplot_kw={"projection": "3d"})
heat.scatter(*voxel.numpy().T, c=scaled, cmap="viridis", s=0.4, linewidths=0, depthshade=False)
heat.scatter(*voxel[query].tolist(), marker="*", s=200, color="red")  # the query point
neighbors.scatter(
    *voxel[top].numpy().T,
    c=[palette[label] for label in segment[top]],
    s=1.5,
    linewidths=0,
    depthshade=False,
)
titles = ["cosine similarity to the starred query point", "the 5000 nearest points, by annotated class"]
for ax, title in zip((heat, neighbors), titles):
    ax.set_box_aspect(np.ptp(voxel.numpy(), axis=0))
    ax.set_title(title, fontsize=10)
    ax.set_axis_off()
plt.show()

![One chair point's cosine similarity to every point of the room, and the 5000 nearest points colored by their annotated class.](../assets/tutorials/query_similarity.png)

The left panel is the whole answer: the chairs light up, the floor and walls stay dark, and nothing selected them except the feature the encoder assigned to each point.

The thousand nearest neighbors of that one point are 100% chair. Push to 5000, the right panel, and purity falls to 98.0%, but look at what the other 99 points are: 83 carry no label at all, and only 16 sit on the floor the chairs stand on. Most of what the search is charged with getting wrong is a region the annotator left blank.

That is **nearest-neighbor label transfer**: annotate one point, propagate to a region. With a handful of clicks per class you have a segmentation of the scene, and the encoder never saw a label.

## Retrieve whole shapes

Same operation one level up: a classifier with its head removed is a shape encoder, and one descriptor per object
turns a dataset into a searchable index.

> This section needs the **ModelNet40** benchmark data. `download=True` fetches the resampled release into
> `DATA_ROOT` on first use (several GB once extracted); the cell checks whether it is already there. Everything above
> this point runs from the committed sample room.

In [ ]:
from torch_pointcloud.datasets import ModelNetNormalResampled
from torch_pointcloud.utils.data import PointCloudDataLoader

DATA_ROOT = Path("data")  # the dataset lives in DATA_ROOT / "ModelNetNormalResampled"
present = (DATA_ROOT / "ModelNetNormalResampled" / "processed").exists()
print("already downloaded:", present)

shape_encoder, shape_info = tp.create_model(
    "pointnet2-ssg.modelnet40.xu-yan",
    task="classification",
    pretrained=True,
    num_classes=0,
    return_info=True,
)
shape_encoder = shape_encoder.eval().to(device)

dataset = ModelNetNormalResampled(
    root=DATA_ROOT, variant="40", train=False, transform=shape_info["transform"], download=not present
)
loader = PointCloudDataLoader(dataset, batch_size=64, num_workers=6)

index, labels = [], []
with torch.inference_mode():
    for item in loader:
        descriptor = shape_encoder(None, item["pos"].to(device), item["batch"].to(device))
        index.append(F.normalize(descriptor, dim=-1).cpu())  # normalize once, at write time
        labels.append(item["label"])

index, labels = torch.cat(index), torch.cat(labels)
print("index:", tuple(index.shape), f"| {index.numel() * index.element_size() / 1e6:.1f} MB float32")

```text
already downloaded: True
index: (2468, 1024) | 10.1 MB float32
```

The whole test split is 2468 objects and its index is 10.1 MB, so the search is one matrix product. Set the diagonal
aside first: an object is always its own nearest neighbor.

In [ ]:
similarity = index @ index.t()
similarity.fill_diagonal_(-1.0)  # never retrieve the query itself
nearest = similarity.argmax(dim=1)
top5 = similarity.topk(5, dim=1).indices

print(f"1-NN retrieval accuracy: {(labels[nearest] == labels).float().mean():.4f}")
print(f"top-5 class purity:      {(labels[top5] == labels[:, None]).float().mean():.4f}")
print(f"5-NN vote accuracy:      {(torch.mode(labels[top5], dim=1).values == labels).float().mean():.4f}")

```text
1-NN retrieval accuracy: 0.8825
top-5 class purity:      0.8545
5-NN vote accuracy:      0.8918
```

88.25% of the split retrieves an object of its own class first, against **92.30%** for the same checkpoint's trained
classification head. The head was fitted to these 40 classes; the search was not told the classes exist.

In [ ]:
names = {value: key for key, value in dataset.class_to_idx.items()}
roles = ("tab:blue", "tab:green", "tab:red")  # the query, a neighbor of its class, a neighbor of another

gallery, titles = [], []
for name in ("airplane", "chair", "cup", "desk"):
    query = int((labels == dataset.class_to_idx[name]).nonzero()[0])
    for rank, found in enumerate([query, *top5[query, :3].tolist()]):
        pos = dataset[found]["pos"][:, [0, 2, 1]]  # the dataset is y-up, the figure is z-up
        role = 0 if rank == 0 else int(labels[found] != labels[query]) + 1
        gallery.append({"pos": pos, "color": roles[role]})
        title = names[int(labels[found])].replace("_", " ")
        titles.append(title if rank == 0 else f"{title}  {float(similarity[query, found]):.2f}")

show_clouds(gallery, titles, point_size=1.5, columns=4, height=3.6)

![Four query objects and the three nearest objects to each, with the neighbors that changed class marked in their own color.](../assets/tutorials/retrieval_gallery.png)

Read the misses, not the hits: a cup retrieves a flower pot first, with the nearest real cup 0.0001 of cosine behind it, and a desk brings back two night stands in its top three. Those are cases where the encoder is right about the geometry and a class list disagrees with it.

In [ ]:
from collections import Counter

missed = (labels[nearest] != labels).nonzero().squeeze(-1)
confusions = Counter((names[int(labels[i])], names[int(labels[nearest[i]])]) for i in missed.tolist())

print(f"{len(missed)} of {len(labels)} objects retrieve a different class")
for (asked, found), count in confusions.most_common(5):
    print(f"  {asked:>12} -> {found:<12} {count}")

```text
290 of 2468 objects retrieve a different class
       dresser -> night_stand  20
         plant -> flower_pot   17
   night_stand -> dresser      15
          desk -> table        13
    flower_pot -> plant        13
```

Every one of the top confusions is a pair that shares a shape. Sampled to 1024 points and stripped of color, a dresser
and a night stand are the same box. Project the descriptors of a few classes onto their top three principal components
and the structure the search runs over shows up directly:

In [ ]:
shown = ["airplane", "chair", "table", "bottle", "dresser", "night_stand"]
ids = torch.tensor([dataset.class_to_idx[name] for name in shown])
lookup = torch.full((len(dataset.class_to_idx),), -1, dtype=torch.long)
lookup[ids] = torch.arange(len(shown))  # palette index of each drawn class, -1 for the rest
keep = lookup[labels] >= 0

descriptors = index[keep]
_, _, components = torch.pca_lowrank(descriptors, center=True, q=3, niter=8)
projected = ((descriptors - descriptors.mean(0)) @ components).numpy()
drawn = lookup[labels[keep]].numpy()

_, ax = plt.subplots(figsize=(7.5, 6.0), subplot_kw={"projection": "3d"})
for rank, name in enumerate(shown):
    ax.scatter(*projected[drawn == rank].T, s=4.0, label=name.replace("_", " "), depthshade=False)
ax.legend(loc="upper left", frameon=False, fontsize=9)
ax.set_axis_off()
plt.show()

![Global descriptors of six ModelNet40 classes projected to three dimensions, each class in its own color.](../assets/tutorials/embedding_map.png)

Airplane holds a region of its own and retrieves its own class for all 100 of its test objects; chair, bottle and table hold theirs at 96%, 95% and 93%. Dresser and night stand run into one another along a single ridge, and they are the two worst classes on the split at 73% and 71%, which is those 35 swapped retrievals drawn. Nothing here is a decision boundary: it is only where the encoder put things, and the search inherits it.

## Practical notes

- **Normalize once, at write time.** Store unit-length rows and every search is a plain matrix product, with cosine
  similarity in $[-1, 1]$ and `argmax` as the nearest neighbor. Normalizing inside the query loop instead is the
  usual source of a silently wrong ranking.
- **Batch the encoding pass.** The 2468 test objects above take 26.8 seconds at batch size 64 on one GPU,
  about 92 objects per second, and most of that is the transform on the dataloader workers, not the model.
- **Persist three things**: the descriptor matrix, the ids it is aligned to, and the checkpoint name plus the
  `info["transform"]` it was built with. A query embedded through a different pipeline is not comparable to the
  index, and nothing about the tensor will tell you that. `float16` halves the file with no measurable effect on the
  ranking at this scale; an approximate index only starts to pay off in the millions of rows.
- **A scene index is per point, so subsample it.** 114 118 rows of 1728 channels is 789 MB for one room. Keep the
  features of a few thousand points per scene, or pool them per instance or per voxel before storing.
- **A linear probe beats raw nearest neighbors as soon as you have some labels.** Nearest-neighbor search weighs every
  channel equally; one trained linear layer on the same frozen descriptors learns which channels matter for your
  classes. That is what the `*-lp` checkpoints are, and the [Feature maps](../models/features.md) guide lists their
  scores.

## Next steps

- [Feature maps](../models/features.md): the same recipe across fourteen pretrained backbones, indoors and on LiDAR.
- [Train a model](05-training.md): when a frozen encoder is not enough and you want to fit one.
- [Understand an indoor scene](09-indoor-scene.md): several models read one room at once.